1. Import Libraries

In [1]:
import numpy as np 
import pandas as pd
from sklearn.model_selection import train_test_split

from sklearn.preprocessing import (
    LabelEncoder,
    OneHotEncoder,
    StandardScaler
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

2. Load DataSet

In [2]:
df = pd.read_csv("../data/WA_Fn-UseC_-HR-Employee-Attrition.csv")
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


3. Check Missing Values

In [3]:
df.isnull().sum()

Age                         0
Attrition                   0
BusinessTravel              0
DailyRate                   0
Department                  0
DistanceFromHome            0
Education                   0
EducationField              0
EmployeeCount               0
EmployeeNumber              0
EnvironmentSatisfaction     0
Gender                      0
HourlyRate                  0
JobInvolvement              0
JobLevel                    0
JobRole                     0
JobSatisfaction             0
MaritalStatus               0
MonthlyIncome               0
MonthlyRate                 0
NumCompaniesWorked          0
Over18                      0
OverTime                    0
PercentSalaryHike           0
PerformanceRating           0
RelationshipSatisfaction    0
StandardHours               0
StockOptionLevel            0
TotalWorkingYears           0
TrainingTimesLastYear       0
WorkLifeBalance             0
YearsAtCompany              0
YearsInCurrentRole          0
YearsSince

4. Check Duplicates

In [4]:
df.duplicated().sum()

# if duplicate found, drop it
#df.drop_duplicates()

np.int64(0)

5. Remove Unnecessary Columns

In [5]:
df.drop("EmployeeNumber", axis = 1, inplace=True)

6. Check Contant Columns

In [6]:
for col in df.columns:
    if df[col].nunique() == 1:
        print(col)

EmployeeCount
Over18
StandardHours


In [7]:
df.drop(
    columns=[
        "EmployeeCount",
        "Over18",
        "StandardHours"
    ],
    inplace = True
)

### Observation:
1. `EmployeeCount`, `Over18`, and `StandardHours` have only one unique value across all records.
2. These are constant columns and do not provide any useful information for prediction.
3. Therefore, they are removed from the dataset.

7. Separate Features and Target

In [8]:
x = df.drop("Attrition", axis = 1)
y = df["Attrition"]

### Key Point

1. **Features (X):** Input variables used to make predictions.
2. **Target (y):** Output variable that the model needs to predict.
3. Separating them helps the model understand what to learn from (X) and what to predict (y).

8. Encode Target Variable

In [9]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

### Key Points
For Target Variable
1. The target variable (y) is the output (e.g., Yes/No).
2. We simply convert:
  - Yes → 1
  - No → 0
3. Here, 0 and 1 are just labels, so no problem.

For Input Variable
1. Input features like Country have no natural order.
2. LabelEncoder may create a false order:
  - India → 0
  - USA → 1
  - France → 2
3. The model may incorrectly think France > USA > India.

**Therefore:**
-  Target (y) → LabelEncoder
-  Input Features (X) → OneHotEncoder (for categorical variables)

9. Identify Numerical and Categorical Columns

In [14]:
numerical_columns = x.select_dtypes(include=["int64"]).columns
categorical_columns = x.select_dtypes(include=["object"]).columns

/var/folders/83/lyk88fr96bq5vfw5v_m3j0n40000gn/T/ipykernel_46260/2031502495.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = x.select_dtypes(include=["object"]).columns


In [15]:
print(numerical_columns)
print(categorical_columns)

Index(['Age', 'DailyRate', 'DistanceFromHome', 'Education',
       'EnvironmentSatisfaction', 'HourlyRate', 'JobInvolvement', 'JobLevel',
       'JobSatisfaction', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked',
       'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction',
       'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear',
       'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole',
       'YearsSinceLastPromotion', 'YearsWithCurrManager'],
      dtype='str')
Index(['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole',
       'MaritalStatus', 'OverTime'],
      dtype='str')


#### why? 

This helps us identify which columns require different preprocessing techniques, such as scaling for numerical features and encoding for categorical features.

10. Create a Preprocessing Pipeline

In [29]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_columns
        ),
        (
            "cat",
            OneHotEncoder(drop="first"),
            categorical_columns
        )
    ]
)

#### Key Points
1. `ColumnTransformer` applies different preprocessing to different column types.
2. `StandardScaler()` is applied to numerical columns.
3. `OneHotEncoder(drop="first")` is applied to categorical columns.
4. `drop="first"` removes one dummy column to avoid the Dummy Variable Trap in linear models.
5. Finally, all transformed columns are combined into a single dataset for model training.

11. Train-Test Split

In [19]:
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify = y
)

In [22]:
print(x_train)

      Age     BusinessTravel  DailyRate              Department  \
1194   47      Travel_Rarely       1225                   Sales   
128    22      Travel_Rarely        594  Research & Development   
810    46      Travel_Rarely        406                   Sales   
478    25      Travel_Rarely        622                   Sales   
491    43  Travel_Frequently       1001  Research & Development   
...   ...                ...        ...                     ...   
1213   23      Travel_Rarely        427                   Sales   
963    38      Travel_Rarely       1009                   Sales   
734    22      Travel_Rarely        217  Research & Development   
1315   36      Travel_Rarely        430  Research & Development   
1292   39  Travel_Frequently        766                   Sales   

      DistanceFromHome  Education    EducationField  EnvironmentSatisfaction  \
1194                 2          4     Life Sciences                        2   
128                  2          1  

In [23]:
print(x_test)

      Age BusinessTravel  DailyRate              Department  DistanceFromHome  \
1061   24     Non-Travel        830                   Sales                13   
891    44  Travel_Rarely       1117  Research & Development                 2   
456    31  Travel_Rarely        688                   Sales                 7   
922    44  Travel_Rarely       1199  Research & Development                 4   
69     36  Travel_Rarely        318  Research & Development                 9   
...   ...            ...        ...                     ...               ...   
1269   43  Travel_Rarely        244         Human Resources                 2   
1352   44  Travel_Rarely        170  Research & Development                 1   
1236   36  Travel_Rarely       1456                   Sales                13   
1023   56  Travel_Rarely       1255  Research & Development                 1   
285    37  Travel_Rarely       1372  Research & Development                 1   

      Education EducationFi

In [27]:
print(y_train)

[0 0 0 ... 0 0 0]


In [25]:
print(y_test)

[0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 1 0 1 0 0 0 0 0 1 0
 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0
 0 0 0 0 0 0 0 1 1 0 0 0 0 1 0 0 0 0 1 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 1 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0
 0 0 0 0 0 0 0 1 1 0 1 0 0 0 1 0 1 0 1 0 1 0 0 0 0 0 1 0 1 0 1 0 0 0 0 1 0
 0 0 0 0 0 1 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 1 0 0
 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 1 0 0 0 1 1 0 0 0 0 1 0 0
 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 0]


#### Key points

The dataset is split into **80% training data** and **20% testing data**.

1. **Training Set:** Used to train the machine learning model.
2. **Testing Set:** Used to evaluate the model on unseen data.
3. **random_state=42:** Ensures the same split every time for reproducibility.
4. **stratify=y:** Maintains the same class distribution in both training and testing sets.

12. Fit the Preprocessor

In [30]:
x_train = preprocessor.fit_transform(x_train)

x_test = preprocessor.transform(x_test)

In [34]:
print(x_train)

[[ 1.09019402  1.04945488 -0.89991452 ...  0.          0.
   0.        ]
 [-1.6348276  -0.52344929 -0.89991452 ...  1.          0.
   0.        ]
 [ 0.98119316 -0.99208001 -0.77761018 ...  1.          0.
   0.        ]
 ...
 [-1.6348276  -1.46320345 -0.16608847 ...  1.          0.
   0.        ]
 [-0.10881549 -0.93225481 -0.89991452 ...  1.          0.
   1.        ]
 [ 0.21818711 -0.09470203  1.30156365 ...  0.          0.
   0.        ]]


In [33]:
print(x_test)

[[-1.41682587e+00  6.48318281e-02  4.45433249e-01 ...  1.00000000e+00
   0.00000000e+00  0.00000000e+00]
 [ 7.63191429e-01  7.80241491e-01 -8.99914522e-01 ...  1.00000000e+00
   0.00000000e+00  0.00000000e+00]
 [-6.53819813e-01 -2.89133929e-01 -2.88392808e-01 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 ...
 [-1.08815489e-01  1.62527242e+00  4.45433249e-01 ...  0.00000000e+00
   0.00000000e+00  1.00000000e+00]
 [ 2.07120181e+00  1.12423638e+00 -1.02221887e+00 ...  1.00000000e+00
   0.00000000e+00  0.00000000e+00]
 [ 1.85375620e-04  1.41588422e+00 -1.02221887e+00 ...  0.00000000e+00
   1.00000000e+00  0.00000000e+00]]


### Important Points

1. fit_transform() is used on the training data to learn preprocessing (e.g., scaling, encoding) and then apply it.

2. transform() is used on the test data to apply the same preprocessing learned from the training data.

3. Never use fit() or fit_transform() on the test data, as it can cause data leakage and lead to incorrect model evaluation.

13. Check Shapes

In [35]:
print(x_train.shape)
print(x_test.shape)

(1176, 44)
(294, 44)


## Summary

### Objective
Prepared the dataset for machine learning by cleaning the data, removing unnecessary features, encoding categorical variables, scaling numerical features, and splitting the data into training and testing sets.

### Tasks Performed
1. Loaded the dataset.
2. Verified missing values and duplicate records.
3. Removed irrelevant and constant columns:
  - `EmployeeNumber`
  - `EmployeeCount`
  - `Over18`
  - `StandardHours`
4. Separated features (`x`) and target (`y`).
5. Encoded the target variable (`Attrition`) using `LabelEncoder`.
6. Identified numerical and categorical features.
7. Created a preprocessing pipeline using:
  - `StandardScaler` for numerical features.
  - `OneHotEncoder` for categorical features.
8. Split the dataset into training and testing sets using stratified sampling.
9. Applied preprocessing to both training and testing data.

### Outcome
The dataset is now clean, properly encoded, scaled, and ready for model training.